# SCRIPT PiNPiS from VCF

## Etape 1: Separer le vcf par population

In [ ]:
#!/bin/bash
#SBATCH --job-name=VCF_split
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=1

module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16


VCF="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/VCF/laurus_snps_final.vcf.gz"
OUTDIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS/INPUT"

mkdir -p "$OUTDIR"

########################
# POPULATIONS
########################

L_azorica_list="LaM16,LaM15,LaF10,LaF14,LaF9,LaF12,LaF8,LaF11,LaF13,LaF5,LaM1,LaF6,LaF2,LaF7,LaF4,LaM3,LaF1,LaM9,LaF15,LaM4,LaM6,LaM17,LaM5,LaM7,LaF3,LaM8,LaM10,LaM11,LaM12,LaF16,LaM13,LaM14"

L_nobilis_POR_list="Lm-M14Cb,Lm-M11Cb,Lm-M09b,Lm-F16b,Lm-F15Cbf,Lm-F13Cbf,Lm-F12Cf,Lm-F09Cf,Lm-F08Cbf,Lm-M15Cb,Lm-F07b,Lm-F05Cbf,Lm-M04b,Lm-F02Cb,Lm-M05b,Lm-M13Cf,Lm-M12Cb,Lm-M07Cf,Lm-F04Cbf,Lm-F06Cbf,Lm-M10b,Lm-F10Cb,Lm-F03Cb,Lm-M08Cb,Lm-F14Cbf,Lm-M06Cb,Lm-M01Cf"

L_nobilis_ITA_list="OR05F,OR03M,OR09M,OR01M,OR04Mb,OR05M,OR01F,OR03Fb,OR04F,OR06Fb,OR10M,OR09F,OR02M,OR07M,OR08F"

L_nobilis_GRE_list="GRE6-7_1,GRE5_2b,GRE3-4_1b,GRE3-4_2b,GRE1-2_5b"

L_nobilis_TUN_list="TUNI06-M,TUNI10-Mb,TUNI13-Fb,TUNI14-Fb,TUNI11-Fb"

L_novocanareinsis_list="LN-T28_F,LN-T27_F,LN-T22_M,LN-T20_M,LN-T18b_M,LN-T17b_M,LN-T23_M,LN-T16_M,LN-T26_F,LN-T24_M,LN-T1_M,LN-T2_F,LN-T3_F,LN-T14_M,LN-T4_M,LN-T11_M,LN-T21_M,LN-T5_F,LN-T6_M,LN-T7_M,LN-T25,LN-T19_M,LN-T10_M,LN-T15_M,LN-T12_M,LN-T13_F"

bcftools annotate -x FORMAT/GL $VCF | \
bcftools view -s $L_azorica_list -v snps -Oz -o $OUTDIR/laurus_snps_final_L_azorica.vcf.gz

bcftools annotate -x FORMAT/GL $VCF | \
bcftools view -s $L_nobilis_POR_list -v snps -Oz -o $OUTDIR/laurus_snps_final_L_nobilis_POR.vcf.gz

bcftools annotate -x FORMAT/GL $VCF | \
bcftools view -s $L_nobilis_ITA_list -v snps -Oz -o $OUTDIR/laurus_snps_final_L_nobilis_ITA.vcf.gz

bcftools annotate -x FORMAT/GL $VCF | \
bcftools view -s $L_nobilis_GRE_list -v snps -Oz -o $OUTDIR/laurus_snps_final_L_nobilis_GRE.vcf.gz

bcftools annotate -x FORMAT/GL $VCF | \
bcftools view -s $L_nobilis_TUN_list -v snps -Oz -o $OUTDIR/laurus_snps_final_L_nobilis_TUN.vcf.gz

bcftools annotate -x FORMAT/GL $VCF | \
bcftools view -s $L_novocanareinsis_list -v snps -Oz -o $OUTDIR/laurus_snps_final_L_novocanareinsis.vcf.gz

########################
# INDEXATION
########################

for file in $OUTDIR/*.vcf.gz
do
    bcftools index -t $file
done

## Etape 2: Obtain chromosome of each individual with the alternative alleles in accordance with a VCF file

In [ ]:
#!/bin/bash
#SBATCH --job-name=VCF_to_fasta_indiv
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=1

module load bioinfo/Bcftools/1.9
module load devel/python/Python-3.11.1 devel/java/17.0.6
module load bioinfo/GATK/4.4.0.0

VCF_DIR=/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS/INPUT
REFERENCE_GENOME=/home/tbertrand/work/Laurus_pop_gen/GENOME/GCA_977007225.1_dmLauAzor1.pri_genomic.fna
VCF_LIST=(
laurus_snps_final_L_azorica.vcf.gz
laurus_snps_final_L_nobilis_POR.vcf.gz
laurus_snps_final_L_nobilis_ITA.vcf.gz
laurus_snps_final_L_nobilis_GRE.vcf.gz
laurus_snps_final_L_nobilis_TUN.vcf.gz
laurus_snps_final_L_novocanareinsis.vcf.gz
)

for VCF in "${VCF_LIST[@]}"
do
    
    POP=${VCF%.vcf.gz}
    mkdir -p "$POP"
    
    bcftools query -l "${VCF_DIR}/${VCF}"  | while read S
    do

    gatk FastaAlternateReferenceMaker -R $REFERENCE_GENOME -O "${POP}/${S}.fasta" --use-iupac-sample "${S}" -V "${VCF_DIR}/${VCF}"

    done
done

# ETAPE 3: Processing_data

## Script 1: `extract_CDS_from_fasta.py`

**Utilisation :**
```bash
python extract_CDS_from_fasta.py \
    --individual LaF10_test \
    --base_path /path/to/data/ \
    --gff /path/to/CDS_only.gff3
```

**Inputs :**
- `individual` : liste d’individus (ex : LaF10_test)
- `base_path` : dossier contenant les fichiers FASTA
- `gff` : fichier GFF3 avec les CDS

**Fichiers attendus (par individu) :**
- `${individual}.fasta`

**Outputs générés :**
Pour chaque individu :
- `${individual}_HEADER.fasta`
- `${individual}_ALL_CHR_FULL_IUPAC_CDS_forward.fasta`
- `${individual}_ALL_CHR_FULL_IUPAC_CDS_reverse.fasta`
- `${individual}_ALL_CHR_FULL_IUPAC_CDS_forward_DIPLOID.fasta`
- `${individual}_ALL_CHR_FULL_IUPAC_CDS_reverse_DIPLOID.fasta`


In [ ]:
import argparse
from Bio import SeqIO

parser = argparse.ArgumentParser(description="Extract CDS and generate diploid sequences")

parser.add_argument("--individual", required=True, help="Individual name")
parser.add_argument("--base_path", required=True, help="Base directory path")
parser.add_argument("--gff", required=True, help="GFF file with CDS")

args = parser.parse_args()

individuals = [args.individual]
base_path = args.base_path
GFF_CDS = args.gff


##############
### SCRIPT ###
##############

for individual in individuals:

    file_ALT = base_path + f"{individual}.fasta"
    new_file_ALT = base_path + f"{individual}_HEADER.fasta"

    with open(new_file_ALT, "w") as new_fasta:
        for record in SeqIO.parse(file_ALT, "fasta"):
            # header brut sans ">"
            full_header = record.description
            # découpe propre
            parts = full_header.split()
            # si ton format est: "52 CDRMVC010000040.1:1-128474"
            prefix = parts[1]          # CDRMVC010000040.1:1-128474
            clean_id = prefix.split(":")[0]
            new_header = f">{clean_id} {prefix}\n"
            new_fasta.write(new_header)
            new_fasta.write(str(record.seq) + "\n")

    dico_chr_ALT={}

    for record in SeqIO.parse(new_file_ALT, "fasta"):
        if str(record.id) not in dico_chr_ALT:
            dico_chr_ALT[str(record.id)]=record.seq

    fasta_file_REVERSE = base_path + f"{individual}_ALL_CHR_FULL_IUPAC_CDS_reverse.fasta"
    fasta_file_FORWARD = base_path + f"{individual}_ALL_CHR_FULL_IUPAC_CDS_forward.fasta"

    count=1
    count_reverse=0
    list_CDS=[]
    with open(fasta_file_REVERSE, "w") as fasta_to_generate_rev:
        with open(fasta_file_FORWARD, "w") as fasta_to_generate_for:
            with open(GFF_CDS, 'r') as gff:
                for line in gff:
                    if line.startswith("#"):
                        continue

                    e=line.split("\t")
                    chromosome=e[0]
                    forward_reverse=e[6]

                    #récupération cds_id + gene_id
                    attributes = e[8]
                    cds_id = "NA"
                    gene_id = "NA"

                    for attr in attributes.split(";"):
                        if attr.startswith("ID="):
                            cds_id = attr.split("=")[1]
                        if attr.startswith("gene_id="):
                            gene_id = attr.split("=")[1]

                    if str(chromosome) in dico_chr_ALT: 
                        start=e[3] 
                        end=e[4] 
                        phase=e[7]
                        sequence=dico_chr_ALT[str(chromosome)][int(start)-1:int(end)]  

                        if str(forward_reverse) == "-":    
                            if str(phase) == "0" : 
                                sequence=sequence.reverse_complement()
                                fasta_to_generate_rev.write(">CDS"+str(count)+"|"+cds_id+"|"+gene_id+"|"+str(chromosome)+"|"+str(start)+"|"+str(end)+"\n"+str(sequence)+"\n")
                            elif str(phase) == "1" :
                                sequence=sequence.reverse_complement()
                                sequence=sequence[1:]
                                fasta_to_generate_rev.write(">CDS"+str(count)+"|"+cds_id+"|"+gene_id+"|"+str(chromosome)+"|"+str(start)+"|"+str(end)+"\n"+str(sequence)+"\n")
                            elif str(phase) == "2" : 
                                sequence=sequence.reverse_complement()
                                sequence=sequence[2:]
                                fasta_to_generate_rev.write(">CDS"+str(count)+"|"+cds_id+"|"+gene_id+"|"+str(chromosome)+"|"+str(start)+"|"+str(end)+"\n"+str(sequence)+"\n")
                                                            
                        if str(forward_reverse) == "+":   
                            if str(phase) == "0" : 
                                fasta_to_generate_for.write(">CDS"+str(count)+"|"+cds_id+"|"+gene_id+"|"+str(chromosome)+"|"+str(start)+"|"+str(end)+"\n"+str(sequence)+"\n")
                            elif str(phase) == "1" : 
                                sequence=sequence[1:]
                                fasta_to_generate_for.write(">CDS"+str(count)+"|"+cds_id+"|"+gene_id+"|"+str(chromosome)+"|"+str(start)+"|"+str(end)+"\n"+str(sequence)+"\n")
                            elif str(phase) == "2" : 
                                sequence=sequence[2:]                                
                                fasta_to_generate_for.write(">CDS"+str(count)+"|"+cds_id+"|"+gene_id+"|"+str(chromosome)+"|"+str(start)+"|"+str(end)+"\n"+str(sequence)+"\n")
                                                                            
                        count+=1


##ETAPE 2

    # Forward CDS:
    file_ALT = base_path + f"{individual}.fasta"
    new_file_ALT = base_path + f"{individual}_HEADER.fasta"

    file_to_duplicate_seq_for = base_path + f"{individual}_ALL_CHR_FULL_IUPAC_CDS_forward.fasta"
    file_duplicated_forward = base_path + f"{individual}_ALL_CHR_FULL_IUPAC_CDS_forward_DIPLOID.fasta"

    no_change=['A','T','G','C']

    count=1
    with open(file_duplicated_forward, "w") as fR:
        for record in SeqIO.parse(str(file_to_duplicate_seq_for), "fasta"):
            sequence_1=""
            sequence_2=""
            id_seq=str(record.id) #>CDS85_chr1_3621451_3621710
            id_seq_e=id_seq.split("|")
            header=str(id_seq_e[1])+"|"+str(id_seq_e[2])+"|"+str(id_seq_e[3])
            SEQ=record.seq
            for car in SEQ:
                if str(car) == ".":
                    sequence_1=str(sequence_1)+"N"
                    sequence_2=str(sequence_2)+"N"
                if str(car) in no_change:
                    sequence_1=str(sequence_1)+str(car)
                    sequence_2=str(sequence_2)+str(car)
                if str(car) == "W":
                    sequence_1=str(sequence_1)+"A"
                    sequence_2=str(sequence_2)+"T"
                if str(car) == "S":
                    sequence_1=str(sequence_1)+"C"
                    sequence_2=str(sequence_2)+"G"
                if str(car) == "M":
                    sequence_1=str(sequence_1)+"A"
                    sequence_2=str(sequence_2)+"C"
                if str(car) == "K":
                    sequence_1=str(sequence_1)+"G"
                    sequence_2=str(sequence_2)+"T"
                if str(car) == "R":
                    sequence_1=str(sequence_1)+"A"
                    sequence_2=str(sequence_2)+"G"
                if str(car) == "Y":
                    sequence_1=str(sequence_1)+"C"
                    sequence_2=str(sequence_2)+"T"
                    
            if  len(sequence_1) == len(sequence_2) and len(sequence_1) == len(SEQ): #check that sequences have the same length, as expected
                #>CDS_1|Pearl_millet|So-21-28371-02|Allele_1
                header_1=">CDS_"+str(count)+"|"+str(header)+"|"+str(individual)+"|Allele_1"
                fR.write(str(header_1)+"\n")
                fR.write(str(sequence_1)+"\n")
                header_2=">CDS_"+str(count)+"|"+str(header)+"|"+str(individual)+"|Allele_2"
                fR.write(str(header_2)+"\n")
                fR.write((sequence_2)+"\n")
            else:
                print(header)
            count+=1

    # Reverse CDS:
    file_to_duplicate_seq_rev = base_path + f"{individual}_ALL_CHR_FULL_IUPAC_CDS_reverse.fasta"
    file_duplicated_reverse = base_path + f"{individual}_ALL_CHR_FULL_IUPAC_CDS_reverse_DIPLOID.fasta"


    count=1
    with open(file_duplicated_reverse, "w") as fR:
        for record in SeqIO.parse(str(file_to_duplicate_seq_rev), "fasta"):
            sequence_1=""
            sequence_2=""
            id_seq=str(record.id) #>CDS85_chr1_3621451_3621710
            id_seq_e=id_seq.split("|")
            header=str(id_seq_e[1])+"|"+str(id_seq_e[2])+"|"+str(id_seq_e[3])
            SEQ=record.seq
            for car in SEQ:
                if str(car) == ".":
                    sequence_1=str(sequence_1)+"N"
                    sequence_2=str(sequence_2)+"N"
                if str(car) in no_change:
                    sequence_1=str(sequence_1)+str(car)
                    sequence_2=str(sequence_2)+str(car)
                if str(car) == "W":
                    sequence_1=str(sequence_1)+"A"
                    sequence_2=str(sequence_2)+"T"
                if str(car) == "S":
                    sequence_1=str(sequence_1)+"C"
                    sequence_2=str(sequence_2)+"G"
                if str(car) == "M":
                    sequence_1=str(sequence_1)+"A"
                    sequence_2=str(sequence_2)+"C"
                if str(car) == "K":
                    sequence_1=str(sequence_1)+"G"
                    sequence_2=str(sequence_2)+"T"
                if str(car) == "R":
                    sequence_1=str(sequence_1)+"A"
                    sequence_2=str(sequence_2)+"G"
                if str(car) == "Y":
                    sequence_1=str(sequence_1)+"C"
                    sequence_2=str(sequence_2)+"T"
            if  len(sequence_1) == len(sequence_2) and len(sequence_1) == len(SEQ): #check that sequences have the same length, as expected
                #>CDS_1|Pearl_millet|So-21-28371-02|Allele_1
                header_1=">CDS_"+str(count)+"|"+str(header)+"|"+str(individual)+"|Allele_1"
                fR.write(str(header_1)+"\n")
                fR.write(str(sequence_1)+"\n")
                header_2=">CDS_"+str(count)+"|"+str(header)+"|"+str(individual)+"|Allele_2"
                fR.write(str(header_2)+"\n")
                fR.write((sequence_2)+"\n")
            else:
                print(header)
            count+=1

## Etape intermediaire bash 1 (recuperer le nb de CDS forward et reverse)

In [ ]:
nb_forward=$(grep ">" ${indiv}_ALL_CHR_FULL_IUPAC_CDS_forward_DIPLOID.fasta | wc -l)
nb_reverse=$(grep ">" ${indiv}_ALL_CHR_FULL_IUPAC_CDS_reverse_DIPLOID.fasta | wc -l)

nb_forward=$((nb_forward / 2))
nb_reverse=$((nb_reverse / 2))

## Script 2: `Split_CDS.py`

**Utilisation :**
```bash
python split_CDS.py \
    --individuals LaF10_test LaF12_test \
    --dir_indiv /data/ \
    --dir_cds /data/CDS/ \
    --nb_forward 93083 \
    --nb_reverse 92134
```

**Inputs :**
- `${individual}_ALL_CHR_FULL_IUPAC_CDS_forward_DIPLOID.fasta`
- `${individual}_ALL_CHR_FULL_IUPAC_CDS_reverse_DIPLOID.fasta`
- `dir_indiv` : dossier contenant ces fichiers
- `dir_cds` : dossier de sortie
- `nb_forward` : nombre de CDS forward  
- `nb_reverse` : nombre de CDS reverse  

**Calcul du nombre de CDS :**
```bash
grep ">" file_forward.fasta | wc -l
```
- même commande pour reverse  
- puis diviser par 2 (données diploïdes)

**Outputs :**
Dans `dir_cds/` :
- `CDS_1_forward.fasta`
- `CDS_2_forward.fasta`
- ...
- `CDS_X_reverse.fasta`

👉 Chaque fichier contient les séquences de tous les individus pour un CDS donné.

In [ ]:
import argparse
from Bio import SeqIO

parser = argparse.ArgumentParser(description="Split CDS into individual files")

parser.add_argument("--individuals", nargs="+", required=True, help="List of individuals")
parser.add_argument("--dir_indiv", required=True, help="Directory with diploid fasta files")
parser.add_argument("--dir_cds", required=True, help="Output directory for CDS")
parser.add_argument("--nb_forward", type=int, required=True, help="Number of forward CDS")
parser.add_argument("--nb_reverse", type=int, required=True, help="Number of reverse CDS")

args = parser.parse_args()

individuals = args.individuals
directory_indiv = args.dir_indiv
directory_CDS = args.dir_cds
nb_CDS_forward = args.nb_forward
nb_CDS_reverse = args.nb_reverse


# =========================
# FORWARD
# =========================
for indiv in individuals:
    file_path = f"{directory_indiv}{indiv}_ALL_CHR_FULL_IUPAC_CDS_forward_DIPLOID.fasta"

    for record in SeqIO.parse(file_path, "fasta"):
        header = record.id
        cds_id = int(header.split("|", 1)[0][4:])

        if 1 <= cds_id <= nb_CDS_forward:
            out_path = f"{directory_CDS}CDS_{cds_id}_forward.fasta"

            with open(out_path, "a") as out:
                out.write(f">{header}\n{record.seq}\n")


# =========================
# REVERSE
# =========================
for indiv in individuals:
    file_path = f"{directory_indiv}{indiv}_ALL_CHR_FULL_IUPAC_CDS_reverse_DIPLOID.fasta"

    for record in SeqIO.parse(file_path, "fasta"):
        header = record.id
        cds_id = int(header.split("|", 1)[0][4:])

        if 1 <= cds_id <= nb_CDS_reverse:
            out_path = f"{directory_CDS}CDS_{cds_id}_reverse.fasta"

            with open(out_path, "a") as out:
                out.write(f">{header}\n{record.seq}\n")

## Etape intermediaire bash (Concatener les CDS)

In [ ]:
find . -name "CDS*_reverse.fasta" | xargs cat > all_CDS_reverse.fasta
# Forward
find . -name "CDS*_forward.fasta" | xargs cat > all_CDS_forward.fasta

## Script 3: `group_by_chromosome.py`

**Utilisation :**
```bash
python group_by_chromosome.py \
    --input_forward all_CDS_forward.fasta \
    --input_reverse all_CDS_reverse.fasta \
    --dir_chr /data/CHROMOSOME/
```

**Inputs :**
- `all_CDS_forward.fasta`
- `all_CDS_reverse.fasta`

Fichiers issus de la concaténation des :
- `CDS_*_forward.fasta`
- `CDS_*_reverse.fasta`

**Outputs :**
Dans `dir_chr/` :
- `${chrom}_forward.fasta`
- `${chrom}_reverse.fasta`

Les séquences sont regroupées par chromosome.

In [ ]:
import argparse
from Bio import SeqIO
import os

parser = argparse.ArgumentParser(description="Group CDS by chromosome")

parser.add_argument("--input_forward", required=True, help="Concatenated forward fasta")
parser.add_argument("--input_reverse", required=True, help="Concatenated reverse fasta")
parser.add_argument("--dir_chr", required=True, help="Output chromosome directory")

args = parser.parse_args()

input_forward = args.input_forward
input_reverse = args.input_reverse
directory_CHR = args.dir_chr

chromosomes = set()

for file in [input_forward, input_reverse]:
    for record in SeqIO.parse(file, "fasta"):
        chrom = record.id.split("|")[3]
        chromosomes.add(chrom)

for chrom in chromosomes:

    # forward
    with open(directory_CHR + f"{chrom}_forward.fasta", "w") as out_f:
        for record in SeqIO.parse(input_forward, "fasta"):
            if record.id.split("|")[3] == chrom:
                out_f.write(">" + record.id + "\n")
                out_f.write(str(record.seq) + "\n")

    # reverse
    with open(directory_CHR + f"{chrom}_reverse.fasta", "w") as out_r:
        for record in SeqIO.parse(input_reverse, "fasta"):
            if record.id.split("|")[3] == chrom:
                out_r.write(">" + record.id + "\n")
                out_r.write(str(record.seq) + "\n")

## Etape intermediaire bash (Concatener forward et reverse par CHR)

In [ ]:
cd /data/CHROMOSOME/

# concat forward + reverse
for f in *_forward.fasta
do
    base=${f%_forward.fasta}
    cat "${base}_forward.fasta" "${base}_reverse.fasta" > "${base}_forward_reverse.fasta"
done

# suppression intermédiaires
for f in *_forward_reverse.fasta
do
    base=${f%_forward_reverse.fasta}
    rm -f "${base}_forward.fasta"
    rm -f "${base}_reverse.fasta"
done

## Script 4: `remove_N.py`

In [ ]:
import argparse
from Bio import SeqIO
import glob
import os

parser = argparse.ArgumentParser(description="Remove sequences containing N")
parser.add_argument("--dir", required=True, help="Directory with chromosome fasta files")

args = parser.parse_args()
directory = args.dir

for file in glob.glob(os.path.join(directory, "*_forward_reverse.fasta")):
    new_file = file.replace(".fasta", "_no_indiv_with_N.fasta")

    with open(new_file, "w") as new:
        for record in SeqIO.parse(file, "fasta"):
            seq = str(record.seq)
            if "N" not in seq:
                new.write(f">{record.id}\n{seq}\n")

## Script 5: `rename_headers.py`

In [ ]:
import argparse
from Bio import SeqIO
import glob
import os

parser = argparse.ArgumentParser(description="Rename fasta headers")
parser.add_argument("--dir", required=True, help="Directory with filtered fasta files")
parser.add_argument("--species_name", required=True, help="Species name to insert in header")

args = parser.parse_args()
directory = args.dir
species_name = args.species_name

for file in glob.glob(os.path.join(directory, "*_no_indiv_with_N.fasta")):
    new_file = file.replace(".fasta", "_renamed.fasta")

    with open(new_file, "w") as out:
        for record in SeqIO.parse(file, "fasta"):
            parts = record.id.split("|")

            new_header = (
                f"{parts[1]}_{parts[2]}_{parts[3]}"
                f"|{species_name}|{parts[4]}|{parts[5]}"
            )

            out.write(f">{new_header}\n{record.seq}\n")

## ORGANISATION DIRECTORIES

In [ ]:
PiNPiS/
│
├── INPUT/
│   ├── VCF/
│   │   ├── L_azorica.vcf.gz
│   │   ├── L_nobilis_GRE.vcf.gz
│   │   └── ...
│
├── WORK/
│   ├── L_azorica/
│   │   ├── 01_Indiv/
│   │   ├── 02_CDS/
│   │   ├── 03_CHR/
│   │   
│   │   
│   │
│   ├── L_nobilis_GRE/
│   ├── L_nobilis_ITA/
│   └── ...
│
├── RESULTS/
│   ├── L_azorica_final/
│   ├── L_nobilis_GRE_final/
│   └── ...
│
├── SCRIPTS/
│   ├── extract_CDS.py
│   ├── split_CDS.py
│   ├── group_chr.py
│   ├── clean.py


## PIPELINE SCRIPT

In [ ]:
#!/bin/bash
#SBATCH --job-name=Processing_PiNPiS
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=1

########################
# MODULES
########################

module load devel/python/Python-3.11.1

########################
# PARAMS
########################

SPECIES=L_azorica

BASE=/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS

WORK_DIR=${BASE}/WORK/${SPECIES}
INDIV_DIR=${WORK_DIR}/01_Indiv/
CDS_DIR=${WORK_DIR}/02_CDS/
CHR_DIR=${WORK_DIR}/03_CHR/
SCRIPTS=${BASE}/SCRIPTS
GFF=${BASE}/INPUT/GFF/${SPECIES}_filtered_CDS.gff3

# individus de cette espèce
INDIVIDUALS="LaM16 LaM15 LaF10 LaF14 LaF9 LaF12 LaF8 LaF11 LaF13 LaF5 LaM1 LaF6 LaF2 LaF7 LaF4 LaM3 LaF1 LaM9 LaF15 LaM4 LaM6 LaM17 LaM5 LaM7 LaF3 LaM8 LaM10 LaM11 LaM12 LaF16 LaM13 LaM14"
INDIV_REF=LaF10   # individu de référence pour les comptes de CDS (juste un indiv de la list random)

########################
# INIT DOSSIERS
########################

mkdir -p ${INDIV_DIR}
mkdir -p ${CDS_DIR}
mkdir -p ${CHR_DIR}

########################
# STEP 1: EXTRACT CDS
########################

echo "=== STEP 1: EXTRACT CDS ==="

for ind in ${INDIVIDUALS}
do
    python ${SCRIPTS}/extract_CDS_from_fasta.py \
        --individual ${ind} \
        --base_path ${INDIV_DIR} \
        --gff ${GFF}
done

echo "=== REMOVE INTERMEDIATE FILES  ==="

rm ${INDIV_DIR}*ALL_CHR_FULL_IUPAC_CDS_forward.fasta
rm ${INDIV_DIR}*ALL_CHR_FULL_IUPAC_CDS_reverse.fasta

########################
# STEP 2: SPLIT CDS
########################

echo "=== STEP 2: SPLIT CDS ==="

nb_forward=$(grep ">" ${INDIV_DIR}${INDIV_REF}_ALL_CHR_FULL_IUPAC_CDS_forward_DIPLOID.fasta | wc -l)
nb_reverse=$(grep ">" ${INDIV_DIR}${INDIV_REF}_ALL_CHR_FULL_IUPAC_CDS_reverse_DIPLOID.fasta | wc -l)

nb_forward=$((nb_forward / 2))
nb_reverse=$((nb_reverse / 2))

python ${SCRIPTS}/Split_CDS.py \
    --individuals ${INDIVIDUALS} \
    --dir_indiv ${INDIV_DIR} \
    --dir_cds ${CDS_DIR} \
    --nb_forward ${nb_forward} \
    --nb_reverse ${nb_reverse}

########################
# CONCAT CDS
########################

echo "=== CONCAT CDS ==="

# FORWARD
for f in ${CDS_DIR}CDS*_forward.fasta
do
    cat "$f"
done > ${CDS_DIR}all_CDS_forward.fasta

# REVERSE
for f in ${CDS_DIR}CDS*_reverse.fasta
do
    cat "$f"
done > ${CDS_DIR}all_CDS_reverse.fasta

########################
# STEP 3: GROUP BY CHR
########################

echo "=== STEP 3: GROUP BY CHR ==="

python ${SCRIPTS}/group_by_chromosome.py \
    --input_forward ${CDS_DIR}all_CDS_forward.fasta \
    --input_reverse ${CDS_DIR}all_CDS_reverse.fasta \
    --dir_chr ${CHR_DIR}

########################
# MERGE FORWARD + REVERSE
########################

echo "=== MERGE FORWARD + REVERSE ==="

for f in ${CHR_DIR}*_forward.fasta
do
    base=${f%_forward.fasta}
    cat "${base}_forward.fasta" "${base}_reverse.fasta" > "${base}_forward_reverse.fasta"
done

########################
# CLEAN INTERMEDIATES
########################

echo "=== CLEAN INTERMEDIATES ==="

for f in ${CHR_DIR}*_forward_reverse.fasta
do
    base=${f%_forward_reverse.fasta}
    rm "${base}_forward.fasta"
    rm "${base}_reverse.fasta"
done

########################
# STEP 4: REMOVE N
########################

echo "=== REMOVE N ==="

python ${SCRIPTS}/remove_N.py --dir ${CHR_DIR}

########################
# STEP 5: RENAME HEADERS
########################

echo "=== RENAME HEADERS ==="

python ${SCRIPTS}/rename_headers.py \
    --dir ${CHR_DIR} \
    --species_name ${SPECIES}

########################
# (OPTIONNEL) CLEAN FINAL
########################

# garder seulement les fichiers finaux
rm  ${CHR_DIR}*_forward_reverse.fasta
rm  ${CHR_DIR}*_no_indiv_with_N.fasta


#################################################
# fichier qui groupe les 11 autosomes
#################################################

ls ${CHR_DIR}OZ*_forward_reverse_no_indiv_with_N_renamed.fasta | grep -v "OZ345845.1" | xargs cat > ${CHR_DIR}Autosomes_forward_reverse_no_indiv_with_N_renamed.fasta

echo "=== PIPELINE DONE FOR ${SPECIES} ==="

# ETAPE 4: Run_PiNPis.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=dNdS_array
#SBATCH --cpus-per-task=1
#SBATCH --mem=12G
#SBATCH --time=02:00:00
#SBATCH --array=0-37%8
#SBATCH -p workq

SPECIES=L_azorica

BASE=/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS
WORK_DIR=${BASE}/WORK/${SPECIES}
CHR_DIR=${WORK_DIR}/03_CHR

FILES=(${CHR_DIR}/*.fasta)
FILE=${FILES[$SLURM_ARRAY_TASK_ID]}

echo "Processing $FILE"

${BASE}/dNdSpiNpiS_1.0 \
    -alignment_file="$FILE" \
    -ingroup=${SPECIES} \
    -gapN_site=10

SCRIPT CDS FILTERING PER SPECIES

In [ ]:
#!/bin/bash
#SBATCH --job-name=cds_callable
#SBATCH -p workq
#SBATCH --time=6:00:00
#SBATCH --mem=16G
#SBATCH --cpus-per-task=2

module load bioinfo/VCFtools/0.1.16
module load devel/python/Python-3.11.1
module load bioinfo/bedtools/2.31.1
module load bioinfo/Bcftools/1.21


########################
# INPUTS
########################

VCF="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/Callable_site_counts/filtered_all.vcf.gz"

GFF="/home/tbertrand/work/Laurus_pop_gen/GENOME/braker.fixed_ids.gff3"

POP_DIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/Pi"

OUTDIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/CDS_POP"

mkdir -p ${OUTDIR}

POPS=(
"L_azorica"
"L_nobilis_POR"
"L_nobilis_ITA"
"L_nobilis_GRE"
"L_nobilis_TUN"
"L_novocanariensis"
)

########################
# LOOP POP
########################

for POP in "${POPS[@]}"; do

    echo "======================"
    echo "Processing ${POP}"
    echo "======================"

    POP_BED=${OUTDIR}/${POP}_positions.bed
    CDS_BED=${OUTDIR}/CDS.bed
    CDS_LIST=${OUTDIR}/${POP}_CDS_list.txt
    OUT_GFF=${OUTDIR}/${POP}_filtered_CDS.gff3

    ####################
    # 1. POSITIONS POP
    ####################

    bcftools view -S ${POP_DIR}/${POP}.txt ${VCF} \
    | bcftools query -f '%CHROM\t%POS\n' \
    | awk 'BEGIN{OFS="\t"} {print $1, $2-1, $2}' \
    > ${POP_BED}

    ####################
    # 2. CDS BED (once)
    ####################


    awk '$3=="CDS" {
        split($9,a,";");
        id="NA";
        for(i in a) if(a[i] ~ /^ID=/) id=substr(a[i],4);
        print $1"\t"$4"\t"$5"\t"id
    }' ${GFF} > ${CDS_BED}
    

    ####################
    # 3. CDS LIST POP
    ####################

    bedtools intersect \
        -a ${CDS_BED} \
        -b ${POP_BED} \
        -wa -u \
    | cut -f4 \
    | sort -u \
    > ${CDS_LIST}

    ####################
    # 4. FILTER ORIGINAL GFF
    ####################

    awk 'NR==FNR {keep[$1]; next} \
    $3=="CDS" {
        split($9,a,";");
        id="";
        for(i in a) if(a[i] ~ /^ID=/) id=substr(a[i],4);
        if(id in keep) print
    }' ${CDS_LIST} ${GFF} \
    > ${OUT_GFF}

done

# ETAPE 5: RESULTS

In [ ]:
scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS/RESULTS/L_azorica /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS

scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS/RESULTS/L_nobilis_POR /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS

scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS/RESULTS/L_nobilis_ITA /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS

scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS/RESULTS/L_nobilis_TUN /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS

scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS/RESULTS/L_nobilis_GRE /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS

scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/PiNPiS/RESULTS/L_novocanariensis /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS

### Concatenation result ALL CHR (Input R)

In [ ]:
#L_azorica
cd /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS/L_azorica
cat OZ*renamed.out > ALL_CHR_forward_reverse_no_indiv_with_N_renamed.out

#L_nobilis_POR
cd /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS/L_nobilis_POR
cat OZ*renamed.out > ALL_CHR_forward_reverse_no_indiv_with_N_renamed.out

#L_nobilis_ITA
cd /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS/L_nobilis_ITA
cat OZ*renamed.out > ALL_CHR_forward_reverse_no_indiv_with_N_renamed.out

#L_nobilis_TUN
cd /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS/L_nobilis_TUN
cat OZ*renamed.out > ALL_CHR_forward_reverse_no_indiv_with_N_renamed.out

#L_nobilis_GRE
cd /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS/L_nobilis_GRE
cat OZ*renamed.out > ALL_CHR_forward_reverse_no_indiv_with_N_renamed.out

#L_novocanariensis
cd /Users/cibio/Desktop/Laurus/Pop_gen/PiNPiS/RESULTS/L_novocanariensis
cat OZ*renamed.out > ALL_CHR_forward_reverse_no_indiv_with_N_renamed.out